In [ ]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import faiss
from sklearn.preprocessing import MinMaxScaler
from rapidfuzz import fuzz, process
import re

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
#loading data
data = pd.read_csv('../anime-dataset-2023.csv')

In [ ]:
pd.set_option('display.max_columns', None)
data.head()

In [ ]:
data.columns

In [ ]:
#removing noise from data
data.drop(columns = ['anime_id', 'Aired', 'Premiered', 'Status', 'Image URL', 'Producers', 'Licensors'],inplace = True)

In [ ]:
# data.dropna(inplace = True)

In [ ]:
data.head()

In [ ]:
#preprocessing data
data['English name'] = data['English name'].str.replace('\xa0', '', regex=False).str.strip()
data['Other name'] = data['Other name'].fillna('')
data = data.drop_duplicates(subset='English name', keep='first').reset_index(drop=True)

In [ ]:
scaler = MinMaxScaler()

In [ ]:
#seperating text and numeric features
data_t = data[[
    'Genres',
    'Synopsis',
    'Type',
    'Studios',
    'Source',
    'Rating'
]]

numeric_cols = [
    'Score',
    'Episodes',
    'Rank',
    'Popularity',
    'Favorites',
    'Scored By'
]


In [ ]:
#this will automatically convert the unknown to NAN
for col in numeric_cols:
    data[col] = pd.to_numeric(
        data[col],
        errors='coerce'
    )

In [ ]:
#loading embeddings or creating it if not present
import os

EMBEDDING_PATH = '../final_embedding.npy'
INDEX_PATH = '../anime.index'

try:
    final_embedding = np.load(EMBEDDING_PATH)
    index = faiss.read_index(INDEX_PATH)
    print("Loaded from cache")
except FileNotFoundError:
    print("Cache not found, recomputing...")
    
    data_t = data_t.fillna(' ')
    data[numeric_cols] = data[numeric_cols].fillna(data[numeric_cols].median())
    data_n = scaler.fit_transform(data[numeric_cols])

    data_t['text'] = (
        data_t['Genres'].str.replace(',', ' ', regex=False) + ' ' +
        data_t['Synopsis'] + ' ' +
        data_t['Type'] + ' ' +
        data_t['Studios'] + ' ' +
        data_t['Source'] + ' ' +
        data_t['Rating'] + ' '
    )

    model = SentenceTransformer(
        'all-Minilm-L6-v2',
        device = device
    )

    movie_embedding = model.encode(
        data_t['text'].tolist(),
        convert_to_numpy = True,
        show_progress_bar = True
    )

    data_n = np.ascontiguousarray(
        data_n.astype(np.float32)
    )
    
    movie_embedding = np.ascontiguousarray(
        movie_embedding.astype(np.float32)
    )

    faiss.normalize_L2(movie_embedding)
    faiss.normalize_L2(data_n)

    final_embedding = np.hstack([
        movie_embedding * 0.4,
        data_n * 0.6
    ])

    final_embedding = final_embedding.astype(np.float32)

    faiss.normalize_L2(final_embedding)
    
    index = faiss.IndexFlatIP(final_embedding.shape[1])
    
    index.add(final_embedding)
    
    np.save(EMBEDDING_PATH, final_embedding)
    faiss.write_index(index, INDEX_PATH)
    print("Saved to cache")
except Exception as e:
    print(e)


In [ ]:
#defining the search range for the fuzz_match_title
data['search_text'] = (
    data['Name'].fillna('') + ' ' +
    data['English name'].fillna('') + ' ' +
    data['Other name'].fillna('')
)
d_title = data['search_text']

In [ ]:
# Resolve user typos and partial anime titles using fuzzy matching
def fuzz_match_title(user_inp, title_list = d_title, threshold = 40):
    result = process.extractOne(
        user_inp.strip().lower(),
        [t.lower() for t in title_list],
        scorer = fuzz.WRatio,
        score_cutoff = threshold
    )
    if result is None:
        return result
    matched_lower, score, idx = result   
    return idx

In [ ]:
#define recommend function
def recommend(user_inp, k=5):
    idx = fuzz_match_title(user_inp)
    if idx is None:
        print(f'No match for {user_inp}')
        return

    query_name = data['English name'][idx]
    # Extract base title (strip trailing numbers, colons, parentheses)
    base_name = re.split(r'[\s:]*(2|II|\(|\:)', query_name)[0].strip().lower()

    query_vec = final_embedding[idx].reshape(1, -1).astype(np.float32)
    faiss.normalize_L2(query_vec)

    # Fetch more candidates to filter from
    S, I = index.search(query_vec, k + 20)

    seen = set()
    results = []
    for i in range(1, len(I[0])):
        candidate_idx = I[0][i]
        candidate_name = data.iloc[candidate_idx]['English name'].lower()
        candidate_base = re.split(r'[\s:]*(2|II|\(|\:)', candidate_name)[0].strip()
        if candidate_base == base_name:
            continue  # skip same franchise
        if candidate_base in seen:
            continue
        seen.add(candidate_base)
        results.append((S[0][i], candidate_idx))
        if len(results) == k:
            break

    print(f'Recommendations for: {query_name}\n')
    for score, cidx in results:
        movie = data.iloc[cidx]['English name'] + ' ' + data.iloc[cidx]['Other name']
        print(f'{movie}\t({score:.4f})\n')

In [ ]:
#evaluation
recommend('Avatar')